In [1]:
from transition import *
from aradi import *
from printing import *
from masterkey import * #takes +- 3minutes, only needed for step 3

In [2]:
r = 9
differential =  [0x00000000800010020000000000000000, 0x10800010000000000000000000000000]

min_weight = 137 # key-averaged probability of main characteristic
bound = 0        # bound on the key-averaged probability of searched characteristics
offset = 0       # offset of the first linear layer used

#### 1) Given a differential, search characteristics (up to a bound)

In [7]:
# get model object and init correlation variable and sign variable lists
model = cp_model.CpModel()
cor_vars = []

# all arbitrary differences
differences = [[(differential[0] >> j)&1 for j in range(127,-1,-1)]] + [[model.NewBoolVar("") for _ in range(128)] for _ in range(r-1)] + [[(differential[1] >> j)&1 for j in range(127,-1,-1)]]

# apply round constraints and collect variables
for i in range(0,r):
    model, ecv = aradi_differential_trails_round(model, differences[i], differences[i+1],i+offset)
    cor_vars += ecv

#Add a maximum if needed (=minimum probability of characteristic taken into account)
model.Add(sum(cor_vars)==min_weight+bound)

# setup callback
cb = TrailCollector3(differences, sum(cor_vars))

# setup solver to enumerate trails
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 2000.0 #search for a set time
solver.parameters.enumerate_all_solutions = True

# solve
solver.Solve(model, cb)

# print out trials
print(cb.number_of_trails())
cb.print_trails()

36
chars = [[0x00000000800000000000000000000000,0x00000000810000080000000000000000,0x848009088581090a0000000000000000,0x0ae7d8c60b6b58c70000000000000000,0xa00a0db0ab8989b00000000000000000,0x9ab044311eb054198000040004001428,0x32840c50100408100000000000000000,0x10040800000000000000000000000000,0x00040000000000000000000000000000,0x08040040000000000000000000000000],
[0x00000000800000000000000000000000,0x00000000810000080000000000000000,0x848009088581090a0000000000000000,0x0ae7d8c60befd8c70000000000000000,0xa00a0db0ab8989b00000000000000000,0x9ab044311eb054198000040004001428,0x32840c50100408100000000000000000,0x10040800000000000000000000000000,0x00040000000000000000000000000000,0x08040040000000000000000000000000],
[0x00000000800000000000000000000000,0x00000000810000080000000000000000,0x848009088581090a0000000000000000,0x0ae7d8c60befd8c70000000000000000,0xa00a0db0ab8989b00000000000000000,0x9ab044311eb054198000040004001428,0x32840c50300408500000000000000000,0x10040800000000000000000000000000,0

#### 2) Given characteristics, search deterministic quasidifferential trails per characteristic and give correlation formula (with key dependency)

In [11]:
# Characteristics to calculate from step 1
chars = [[0x00000000800010020000000000000000,0x00000000000000020000000000000000,0x00000000100008020000000000000000,0x1220081192200c130000000000000000,0xbb365a3fb9365e3f0000000000000000,0xd8e0d445dad1b0450000400000000000,0x30384121301841200000000000000000,0x10100020000000000000000000000000,0x10000000000000000000000000000000,0x10800010000000000000000000000000]]

In [4]:
for i in range(len(chars)):
    characteristic = chars[i]
    print("quasidifferential trails of differential characteristic",i)

    base = []
    correlations = []
    signs = []
    finished = False

    while not finished:
        # get model object and init correlation variable and sign variable lists
        model = cp_model.CpModel()
        cor_vars = []
        sign_vars = []

        # masks (input and output masks are always 0) and differences
        masks = [[0]*128] + [[model.NewBoolVar("") for _ in range(128)] for _ in range(r-1)] + [[0]*128]
        differences = [[(d >> j)&1 for j in range(127,-1,-1)] for d in characteristic]

        # Add constraint that it is in the quotient space of the space that we already found
        model = constraint_quotient_space(base,signs, masks,model,r)

        # apply round constraints and collect variables
        for z in range(0,r):
            model, ecv, esv = aradi_quasidifferential_trails_round_layer(model, masks[z], differences[z], masks[z+1], differences[z+1],z+offset)
            cor_vars += ecv
            sign_vars += esv

        #Add a maximum if needed (=minimum correlation of trail taken into account)
        model.Add(sum(cor_vars)==min_weight)

        model.Add(sum(sum(mask) for mask in masks) > 0)

        # setup solver to enumerate trails
        solver = cp_model.CpSolver()

        # solve
        status = solver.Solve(model)

        # depending on the status of the solver, print the results and a message
        if status == cp_model.OPTIMAL:
            base.append(value_masks(masks,solver))
            if solver.Value(sum(sign_vars))%2 == 0:
                sign = ""
                signs.append(0)
            else:
                sign = "-"
                signs.append(1)
            correlation = f"{sign}2^{{{-solver.Value(sum(cor_vars))}}}"
            correlations.append(correlation)
            print(hex(base[len(base)-1]))
            print_correlation_subkeys(correlation,base[len(base)-1],r) 
            #print_correlation_masterkey(correlation,base[len(base)-1],r)
        else:
            print("Full base found")
            finished = True

        del model, solver, masks, cor_vars, sign_vars

quasidifferential trails of differential characteristic 0
0x1000080000180000000000000000000000000000000001000000000000000000080000002800400000000000000000000040000000000000000000000000000000000000000000000000000000000000000000000000000
(-1)^{k^4_{71} + k^4_{88} + k^4_{111} + k^4_{112} + k^5_{123} + k^6_{72} + k^6_{102} + k^6_{104} + k^6_{117} + k^7_{77}} -2^{-157}
0x180000000000000000000000000000000001000000000000000000080000002800400000000000000000000040000000000000000000000000000000000000000000000000000000000000000000000000000
(-1)^{k^4_{111} + k^4_{112} + k^5_{123} + k^6_{72} + k^6_{102} + k^6_{104} + k^6_{117} + k^7_{77}} 2^{-157}
0x80000002800400000000000000000000040000000000000000000000000000000000000000000000000000000000000000000000000000
(-1)^{k^6_{72} + k^6_{102} + k^6_{104} + k^6_{117} + k^7_{77}} -2^{-157}
0x2800400000000000000000000040000000000000000000000000000000000000000000000000000000000000000000000000000
(-1)^{k^6_{102} + k^6_{104} + k^6_{117} + k^7_{77}} 2^{-157}
0x80

In [1]:
for b in range(len(chars)):
    characteristic = chars[b]

    base = []
    correlations = []
    signs = []
    finished = False

    while not finished:

        # get model object and init correlation variable and sign variable lists
        model = cp_model.CpModel()
        cor_vars = []
        sign_vars = []

        # masks (input and output masks are always 0) and differences
        masks = [[0]*128] + [[model.NewBoolVar("") for _ in range(128)] for _ in range(r-1)] + [[0]*128]
        differences = [[(d >> j)&1 for j in range(127,-1,-1)] for d in characteristic]

        # Add constraint that it is in the quotient space of the space that we already found
        model = constraint_quotient_space(base,signs, masks,model,r)

        # apply round constraints and collect variables
        for i in range(0,r):
            model, ecv, esv = aradi_quasidifferential_trails_round_layer(model, masks[i], differences[i], masks[i+1], differences[i+1],i+offset)
            cor_vars += ecv
            sign_vars += esv

        #Add a maximum if needed (=minimum correlation of trail taken into account)
        model.Add(sum(cor_vars)==min_weight)

        model.Add(sum(sum(mask) for mask in masks) > 0)

        # setup solver to enumerate trails
        solver = cp_model.CpSolver()

        # solve
        status = solver.Solve(model)

        # depending on the status of the solver, print the results and a message
        if status == cp_model.OPTIMAL:
            base.append(value_masks(masks,solver))
            if solver.Value(sum(sign_vars))%2 == 0:
                signs.append(0)
            else:
                signs.append(1)
        else:
            finished = True
        
        del model, solver, masks, cor_vars, sign_vars
    
    print("base_",b," = [" + ",".join(hex(b) for b in base) + "]",sep="")
    print("signs_",b," = ",signs,sep="")

NameError: name 'chars' is not defined

#### 3) Probability Calcultions combining multiple characteristics

In [15]:
# other examples can be found in outputs.ipynb
# 9-round differential of [BFG+25], has 8 characteristics with p_avg=2**-137
cor="2^(-137)"
nb_rounds=9
weight=137
nb_chars=8
base_0  = [0x10002020100020201000202010002020000000000000000000000000000000000000000000000000020000100000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000,0x20204000000000000000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000,0x10000000000000000000000000020000100000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000,0x20000000000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000,0x200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000,0x206080c0200040000000000000000000a1130401210004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x40808000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20404000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x408080000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80120000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1080840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x210104000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20200040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x204040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x821000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x21000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10100020000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x8001010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000]
signs_0 = [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1]
base_1  = [0x10002020100020201000202010002020000000000000000000000000000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20204000000000000000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000,0x40808000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x408080000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2020004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20404000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80120000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1080840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x210104000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20200040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x204040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x821000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x21000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10100020000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x8001010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000]
signs_1 = [1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1]
base_2  = [0x10002020100020201000202010002020000000000000000000000000000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20204000000000000000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000,0x40808000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x408080000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80120000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2020004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20404000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1080840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x210104000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20200040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x204040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x821000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x21000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10100020000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x8001010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000]
signs_2 = [1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1]
base_3  = [0x10002020100020201000202010002020000000000000000000000000000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20204000000000000000000000000000000000000000000000000000000000000020000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000,0x40808000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x408080000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2020004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20404000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80120000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20200040000000000000000821000400080840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x204040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1080840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x210104000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x821000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x21000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10100020000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x8001010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000]
signs_3 = [1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1]
base_4  = [0x10002020100020201000202010002020000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x8001010000000000000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000,0x40808000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x408080000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80120000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x800010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x21010400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1080840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x210004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x204040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20404000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000]
signs_4 = [0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
base_5  = [0x10002020100020201000202010002020000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x8001010000000000000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000,0x40808000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x408080000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80120000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x800010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x21010400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1080840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x210004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x204040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20404000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000]
signs_5 = [0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
base_6  = [0x10002020100020201000202010002020000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x8001010000000000000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000020000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000,0x40808000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x408080000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80120000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x800010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x21010400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1080840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x210004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x204040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20404000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000]
signs_6 = [0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
base_7  = [0x10002020100020201000202010002020000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000,0x8001010000000000000000000000000000000010000000000000000000000000020000100000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000,0x10000000000000000000000000020000100000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000,0x20000000000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000,0x1000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000,0x40808000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x408080000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80120000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x800010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x21010400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x1080840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x80840000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x210004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20004000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x204040000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x20404000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,0x2000400000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000]
signs_7 = [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]

base = [base_0,base_1,base_2,base_3,base_4,base_5,base_6,base_7]
signs = [signs_0,signs_1,signs_2,signs_3,signs_4,signs_5,signs_6,signs_7]

In [ ]:
all_rref = []
all_pivots = []

for b, s in zip(base, signs):
    r, p = row_echelon_form(b, s, nb_rounds)
    all_rref.append(r)
    all_pivots.append(p)

full_base = [elem for b in base for elem in b]
full_signs = [s for signs_i in signs for s in signs_i]
full_rref, full_pivots = row_echelon_form(full_base,full_signs,nb_rounds)
print("The amount of independent base elements:", len(full_pivots))
print("Does the same trails with opposite signs exist?",full_pivots[len(full_pivots)-1]," if=128*(r+1), then yes")
print(full_pivots)
pivot_sets = [set(p) for p in all_pivots]
common = tuple(set.intersection(*pivot_sets))
all_only = [tuple(p - set(common)) for p in pivot_sets]
other = tuple(p for p in full_pivots if p not in common)

print("Number of common elements: ",len(common))
print("When we have opposite sign trails, we have to check them amongst the different basises")
trail_index=0
for c in common:
    print("trail ",trail_index," with pivot ",c)
    index=0
    for rref,pivots in zip(all_rref,all_pivots):
        print_correlation_subkeys_with_sign_numpy([rref[pivots.index(c),:]],cor)
        index+=1
    trail_index +=1

for c in other:
    print("trail ",trail_index," with pivot ",c)
    index=0
    for rref,pivots in zip(all_rref,all_pivots):
        if c not in pivots:
            print("-")
        else:
            print_correlation_subkeys_with_sign_numpy([rref[pivots.index(c),:]],cor)
            index+=1
    trail_index +=1


print("Now the printing for the paper")   
only_rows = [all_rref[0][all_pivots[0].index(c), :] for c in common]
print_correlation_masterkey_check(only_rows)

NameError: name 'row_echelon_form' is not defined

In [ ]:
all_rref = []
all_pivots = []

for b, s in zip(base, signs):
    r, p = row_echelon_form(b, s, nb_rounds)
    all_rref.append(r)
    all_pivots.append(p)

full_base = np.hstack(base)
full_signs = np.hstack(signs)
full_rref, full_pivots = row_echelon_form(full_base,full_signs,nb_rounds)


pivot_sets = [set(p) for p in all_pivots]
common = tuple(set.intersection(*pivot_sets))
other = tuple(p for p in full_pivots if p not in common)

table = []

trail_index=0
for c in common:
    trail = []
    index=0
    print("trail ",trail_index,end= " & ")
    for rref,pivots in zip(all_rref,all_pivots):
        M = [rref[pivots.index(c),:]]
        a = "+"
        row = M[len(M)-1]
        if row[len(row)-1]==1:
            a = "-"
        trail.append(a)
        print(a,end=" & ")
        index+=1  
    table.append(trail)
    print("\\\\")
    trail_index +=1

for c in other:
    trail = []
    index=0
    print("trail ",trail_index,end= " & ")
    if c != 128*(nb_rounds+1): # if pivot = 128*(nb_rounds+1) -> don't do anything is not an extra trail but the pivot to show that there's a trail that exists in all char but with different sign
        for rref,pivots in zip(all_rref,all_pivots):
            if c not in pivots:
                a = " "
            else:
                M = [rref[pivots.index(c), :]]
                row = M[len(M)-1]
                if row[len(row)-1]==1:
                    a = "-"
                else:
                    a = "+"
            trail.append(a)
            print(a,end=" & ")
            index+=1  
        table.append(trail)
        print("\\\\")
        trail_index +=1

table = compress_table(table)
print()
print("table = ",table)

NameError: name 'row_echelon_form' is not defined